# **Import Libraries**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix


# **Load Datasets**

In [3]:
df_recommend = pd.read_csv('/content/drive/MyDrive/Crop Minor Project/Crop_recommendation.csv')
df_yield = pd.read_csv('/content/drive/MyDrive/Crop Minor Project/crop_yield.csv')

print("Crop Recommendation Columns:", df_recommend.columns)
print("Crop Yield Columns:", df_yield.columns)


Crop Recommendation Columns: Index(['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall', 'label'], dtype='object')
Crop Yield Columns: Index(['Crop', 'Crop_Year', 'Season', 'State', 'Area', 'Production',
       'Annual_Rainfall', 'Fertilizer', 'Pesticide', 'Yield'],
      dtype='object')


## **Data Cleaning**

In [4]:
# Removing duplicates
df_recommend.drop_duplicates(inplace=True)
df_yield.drop_duplicates(inplace=True)

# Filling missing values with column mean for all numeric columns
df_recommend.fillna(df_recommend.select_dtypes(include=np.number).mean(), inplace=True)
df_yield.fillna(df_yield.select_dtypes(include=np.number).mean(), inplace=True)

# **Merge Datasets Fully on Crop**
# label - crop_recommendation
# Crop - crop_yield

In [5]:
df_recommend['label'] = df_recommend['label'].str.strip().str.lower()
df_yield['Crop'] = df_yield['Crop'].str.strip().str.lower()

merged = pd.merge(
    df_recommend, df_yield,
    left_on='label', right_on='Crop', how='outer', indicator=True
)

# Fill missing values
num_cols = merged.select_dtypes(include=['number']).columns
cat_cols = merged.select_dtypes(include=['object']).columns

merged[num_cols] = merged[num_cols].fillna(merged[num_cols].mean())
merged[cat_cols] = merged[cat_cols].fillna('unknown')

# NOW: Remove placeholder/unknown rows
merged = merged[merged['label'] != 'unknown']

print("Merged dataset shape:", merged.shape)
print("Unique crops in merged dataset:", merged['label'].nunique())
print("Crop labels:", merged['label'].unique())
merged.head()



Merged dataset shape: (278700, 19)
Unique crops in merged dataset: 22
Crop labels: ['apple' 'banana' 'blackgram' 'chickpea' 'coconut' 'coffee' 'cotton'
 'grapes' 'jute' 'kidneybeans' 'lentil' 'maize' 'mango' 'mothbeans'
 'mungbean' 'muskmelon' 'orange' 'papaya' 'pigeonpeas' 'pomegranate'
 'rice' 'watermelon']


,N,P,K,temperature,humidity,ph,rainfall,label,Crop,Crop_Year,Season,State,Area,Production,Annual_Rainfall,Fertilizer,Pesticide,Yield,_merge
0,24.0,128.0,196.0,22.750888,90.694892,5.521467,110.431786,apple,unknown,2008.883247,unknown,unknown,420783.472113,1.061404e+08,1594.340908,5.668353e+07,113914.019443,511.3002,left_only
1,7.0,144.0,197.0,23.849401,94.348150,6.133221,114.051249,apple,unknown,2008.883247,unknown,unknown,420783.472113,1.061404e+08,1594.340908,5.668353e+07,113914.019443,511.3002,left_only
2,14.0,128.0,205.0,22.608010,94.589006,6.226290,116.039659,apple,unknown,2008.883247,unknown,unknown,420783.472113,1.061404e+08,1594.340908,5.668353e+07,113914.019443,511.3002,left_only
3,8.0,120.0,201.0,21.186674,91.134357,6.321152,122.233323,apple,unknown,2008.883247,unknown,unknown,420783.472113,1.061404e+08,1594.340908,5.668353e+07,113914.019443,511.3002,left_only
4,20.0,129.0,201.0,23.410447,91.699133,5.587906,116.077793,apple,unknown,2008.883247,unknown,unknown,420783.472113,1.061404e+08,1594.340908,5.668353e+07,113914.019443,511.3002,left_only


In [6]:
print(merged['label'].unique())

['apple' 'banana' 'blackgram' 'chickpea' 'coconut' 'coffee' 'cotton'
 'grapes' 'jute' 'kidneybeans' 'lentil' 'maize' 'mango' 'mothbeans'
 'mungbean' 'muskmelon' 'orange' 'papaya' 'pigeonpeas' 'pomegranate'
 'rice' 'watermelon']


# **Select relevant feature**


*   Location(state)
*   Season

*   Annual Rainfall
*   Temperature






In [7]:
# Encode categorical columns
le_state = LabelEncoder()
merged['State_enc'] = le_state.fit_transform(merged['State'])

le_season = LabelEncoder()
merged['Season'] = merged['Season'].str.strip() # Strip whitespace from season names
le_season.fit(merged['Season']) # Re-fit the LabelEncoder on the cleaned season names
merged['Season_enc'] = le_season.transform(merged['Season'])

# Prepare feature matrix (X) and label vector (y)
X = merged[['State_enc', 'Season_enc', 'Annual_Rainfall', 'temperature']]
y = merged['label']

# Encode labels (crops)
le_crop = LabelEncoder()
y = le_crop.fit_transform(y)

# **Data Validation**
Tempearture and Rainfall Ranges

In [8]:
print("Minimum temperature:", merged['temperature'].min())
print("Maximum temperature:", merged['temperature'].max())
print("Min rainfall:", merged['Annual_Rainfall'].min())
print("Max rainfall:", merged['Annual_Rainfall'].max())

# Also check for NaN or missing values
print("Missing rainfall values:", merged['Annual_Rainfall'].isna().sum())
print("Missing temperature values:", merged['temperature'].isna().sum())


Minimum temperature: 8.825674745
Maximum temperature: 43.67549305
Min rainfall: 301.3
Max rainfall: 6552.7
Missing rainfall values: 0
Missing temperature values: 0


In [9]:
# Clip rainfall to valid range (300 - 6550 mm)
merged['Annual_Rainfall'] = merged['Annual_Rainfall'].clip(lower=300, upper=6550)
# Clip temperature to valid range (7 - 45 °C)
merged['temperature'] = merged['temperature'].clip(lower=7, upper=45)

In [10]:
def validate_temperature(temp):
    if not isinstance(temp, (int, float)):
        print("Temperature must be a number.")
        return False
    if temp < 7 or temp > 45:
        print("Temperature must be between 7°C and 45°C.")
        return False
    return True

def validate_rainfall(rain):
    if not isinstance(rain, (int, float)):
        print("Rainfall must be a number.")
        return False
    if rain < 300 or rain > 6550:
        print("Annual rainfall must be between 300mm and 6550mm.")
        return False
    return True

# Apply validation, explicitly converting to numeric and handling potential NaNs from fillna
merged['temperature'] = merged['temperature'].astype(float).apply(validate_temperature)
merged['Annual_Rainfall'] = merged['Annual_Rainfall'].astype(float).apply(validate_rainfall)

# **Split Data Into Train/Test Sets**

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# **Train the Random Forest Classifier**

In [12]:
rf_model = RandomForestClassifier(n_estimators=150, random_state=42, class_weight='balanced')
rf_model.fit(X_train, y_train)


RandomForestClassifier(class_weight='balanced', n_estimators=150,
                       random_state=42)

# **Evaluate Model Performance**

In [13]:
y_pred = rf_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


Accuracy: 0.8797273053462504
Classification Report:
               precision    recall  f1-score   support

           0       0.47      0.33      0.39        24
           1       0.91      0.90      0.91      4811
           2       0.16      0.16      0.16        19
           3       0.38      0.50      0.43        24
           4       0.90      0.90      0.90      3463
           5       0.13      0.18      0.15        17
           6       0.41      0.41      0.41        17
           7       0.11      0.04      0.06        23
           8       0.73      0.76      0.74      3618
           9       0.00      0.00      0.00        21
          10       0.04      0.06      0.05        17
          11       0.88      0.88      0.88     19429
          12       0.41      0.41      0.41        17
          13       0.11      0.06      0.08        17
          14       0.17      0.31      0.22        13
          15       0.20      0.19      0.19        16
          16       0.08     

# **Create the Crop Recommendation Function**
Farmer inputs: location, season, annual rainfall, temperature (validated).

In [14]:
def recommend_crop(location, season, annual_rainfall, temperature):

    # Run validation checks
    if not validate_temperature(temperature) or not validate_rainfall(annual_rainfall):
        print("Invalid input values. Please check and try again.")
        return  # Stop execution entirely, no prediction

    # Available encoder classes
    state_classes = list(le_state.classes_)
    season_classes = list(le_season.classes_)

    # Handle unrecognized location
    if location not in state_classes:
        print(f"'{location}' is not recognized in the dataset. Please enter a valid state.")
        return  # stop further execution without predicting

    # Handle unrecognized season
    if season not in season_classes:
        print(f"'{season}' is not recognized in the dataset. Please enter a valid season.")
        return  # stop further execution without predicting

    # Encode categorical values
    loc_enc = le_state.transform([location])[0]
    sea_enc = le_season.transform([season])[0]

    # Prepare features and predict
    features = np.array([[loc_enc, sea_enc, annual_rainfall, temperature]])
    prediction = rf_model.predict(features)
    crop_name = le_crop.inverse_transform(prediction)

    return crop_name[0]


In [15]:
print("Seasons recognized by model:", le_season.classes_)


Seasons recognized by model: ['Autumn' 'Kharif' 'Rabi' 'Summer' 'Whole Year' 'Winter' 'unknown']


In [16]:
print("States recognized by model:", le_state.classes_)


States recognized by model: ['Andhra Pradesh' 'Arunachal Pradesh' 'Assam' 'Bihar' 'Chhattisgarh'
 'Delhi' 'Goa' 'Gujarat' 'Haryana' 'Himachal Pradesh' 'Jammu and Kashmir'
 'Jharkhand' 'Karnataka' 'Kerala' 'Madhya Pradesh' 'Maharashtra' 'Manipur'
 'Meghalaya' 'Mizoram' 'Nagaland' 'Odisha' 'Puducherry' 'Punjab' 'Sikkim'
 'Tamil Nadu' 'Telangana' 'Tripura' 'Uttar Pradesh' 'Uttarakhand'
 'West Bengal' 'unknown']


In [17]:
# Example crop recommendation
print(recommend_crop('Maharashtra', 'Summer', 1400, 30))


banana


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [18]:
print(recommend_crop('Andhra Pradesh', 'Whole Year', 1500, 29))


banana


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [19]:
print(recommend_crop('Rajasthan', 'kharif', 400, 36))


'Rajasthan' is not recognized in the dataset. Please enter a valid state.
None


In [20]:
print(recommend_crop('Kerala', 'kharif', 1200, 100))


Temperature must be between 7°C and 45°C.
Invalid input values. Please check and try again.
None


In [21]:
print(recommend_crop('Punjab', 'Rabi', 700, 19))


maize


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
